In [ ]:
# ---------------
# Dependencies
# ---------------

import torch
import random
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

import pandas
import numpy
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    classification_report,
    average_precision_score,
)

In [ ]:
# ------------------
# Reproducibility
# ------------------


def set_seed(seed: int):
    random.seed(seed)
    numpy.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


seed = 50
set_seed(seed)

In [ ]:
# ------------------------------------
# Device Setup (Is CUDA available?)
# ------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device: ", DEVICE)

In [ ]:
# -----------------------------------
# Load Dataset from GDrive (Colab)
# -----------------------------------

from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# ---------------
# Load dataset
# ---------------

# df = pandas.read_csv("/content/drive/MyDrive/data/dataset.csv")

df = pandas.read_csv("/content/dataset.csv")

if "hash" in df.columns:
    df = df.drop(columns=["hash"])

# Balanced Dataset:
# df_majority = df[df["malware"] == 1]
# df_minority = df[df["malware"] == 0]

# df_majority_down = df_majority.sample(n=len(df_minority), random_state=42)
# df_balanced = pandas.concat([df_majority_down, df_minority]).sample(frac=1, random_state=42)

# df = df_balanced

In [ ]:
# -------------
# Data Setup
# -------------

X = df.drop(columns=["malware"]).values.astype(numpy.float32)
y = df["malware"].values.astype(numpy.int64)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.30, stratify=df["malware"], random_state=42
)

X_train = torch.tensor(X_train).to(DEVICE)
X_val = torch.tensor(X_val).to(DEVICE)

y_train = torch.tensor(y_train).to(DEVICE)
y_val = torch.tensor(y_val).to(DEVICE)

In [ ]:
train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=128,
)

In [ ]:
VOCAB_SIZE = 307
SEQ_LEN = 100
NUM_CLASSES = 2
LATENT_DIM = 128
EMB_DIM = 128
BATCH_SIZE = 128
EPOCHS = 100
HIDDEN_DIM = 256

In [ ]:
class LSTMDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(VOCAB_SIZE, EMB_DIM)

        self.lstm = nn.LSTM(
            input_size=EMB_DIM,
            hidden_size=HIDDEN_DIM,
            num_layers=2,
            dropout=0.3,
            batch_first=True,
        )

        self.fc = nn.Sequential(
            nn.Linear(HIDDEN_DIM, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.4),
            nn.Linear(128, NUM_CLASSES + 1),
        )

    def forward(self, x, embedded=False, return_features=False):
        if not embedded:
            x = x.long()
            x = self.embedding(x)

        output, (hidden, cell) = self.lstm(x)
        features = hidden[-1]
        logits = self.fc(features)

        if return_features:
            return logits, features

        return logits

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()

        self.init_fc = nn.Linear(LATENT_DIM, 256)

        self.rnn = nn.GRU(input_size=EMB_DIM, hidden_size=256, batch_first=True)

        self.token_proj = nn.Linear(256, VOCAB_SIZE)
        self.start_token = nn.Parameter(torch.zeros(1, 1, EMB_DIM))

    def forward(self, z, temperature=0.5):
        batch_size = z.size(0)
        h0 = torch.tanh(self.init_fc(z)).unsqueeze(0)
        inputs = self.start_token.repeat(batch_size, SEQ_LEN, 1)
        outputs, _ = self.rnn(inputs, h0)
        logits = self.token_proj(outputs)
        return F.gumbel_softmax(logits, tau=temperature, hard=True)

In [ ]:
D = LSTMDiscriminator().to(DEVICE)
G = Generator().to(DEVICE)

optimizer_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
optimizer_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))

In [ ]:
for epoch in range(EPOCHS):

    D.train()
    G.train()

    for real_x, real_y in train_loader:

        real_x = real_x.to(DEVICE)
        real_y = real_y.to(DEVICE)
        batch_size = real_x.size(0)

        """Discriminator Training"""
        optimizer_D.zero_grad()

        # Real
        logits_real = D(real_x)
        loss_real = F.cross_entropy(logits_real, real_y)

        # Fake
        z = torch.randn(batch_size, LATENT_DIM).to(DEVICE)
        fake_probs = G(z)

        # Soft tokens to embedding space
        fake_emb = torch.matmul(fake_probs, D.embedding.weight)

        logits_fake = D(fake_emb, embedded=True)

        fake_labels = torch.full((batch_size,), NUM_CLASSES, device=DEVICE)

        loss_fake = F.cross_entropy(logits_fake, fake_labels)

        loss_D = loss_real + loss_fake
        loss_D.backward()
        optimizer_D.step()

        """Generator Training"""
        optimizer_G.zero_grad()

        z = torch.randn(batch_size, LATENT_DIM).to(DEVICE)
        fake_probs = G(z)

        fake_emb = torch.matmul(fake_probs, D.embedding.weight)

        # Get features from discriminator
        logits_fake, feat_fake = D(fake_emb, embedded=True, return_features=True)
        _, feat_real = D(real_x, return_features=True)

        # Feature matching loss
        loss_G = F.mse_loss(feat_fake.mean(dim=0), feat_real.mean(dim=0))

        loss_G.backward()
        optimizer_G.step()

    print(f"Epoch {epoch+1} | D {loss_D:.4f} | G {loss_G:.4f}")

In [ ]:
torch.save(D.state_dict(), f"lstm_sgan_seed_{seed}.pt")

In [ ]:
D.load_state_dict(torch.load(f"lstm_sgan_seed_20.pt"))
D.eval()

with torch.no_grad():
    logits = D(X_val)
    probs = F.softmax(logits[:, :NUM_CLASSES], dim=1)
    preds = torch.argmax(probs, dim=1)

print(classification_report(y_val.cpu().numpy(), preds.cpu().numpy(), digits=4))

print(
    "PR-AUC:", average_precision_score(y_val.cpu().numpy(), probs[:, 1].cpu().numpy())
)

print("ROC-AUC:", roc_auc_score(y_val.cpu().numpy(), probs[:, 1].cpu().numpy()))

1h 3m

seed 10
```
             precision    recall  f1-score   support

           0     0.9077    0.7593    0.8269       324
           1     0.9939    0.9981    0.9960     12839

    accuracy                         0.9922     13163
   macro avg     0.9508    0.8787    0.9114     13163
weighted avg     0.9918    0.9922    0.9918     13163

PR-AUC: 0.9992343652010374
ROC-AUC: 0.9797636252967665
```

seed 20
```
              precision    recall  f1-score   support

           0     0.9301    0.7809    0.8490       324
           1     0.9945    0.9985    0.9965     12839

    accuracy                         0.9932     13163
   macro avg     0.9623    0.8897    0.9227     13163
weighted avg     0.9929    0.9932    0.9929     13163

PR-AUC: 0.9995364487012276
ROC-AUC: 0.9860296175137674
```

seed 30
```

              precision    recall  f1-score   support

           0     0.9078    0.7901    0.8449       324
           1     0.9947    0.9980    0.9963     12839

    accuracy                         0.9929     13163
   macro avg     0.9513    0.8940    0.9206     13163
weighted avg     0.9926    0.9929    0.9926     13163

PR-AUC: 0.999426550657775
ROC-AUC: 0.981292291330716
```

seed 40

```
              precision    recall  f1-score   support

           0     0.9204    0.8210    0.8679       324
           1     0.9955    0.9982    0.9968     12839

    accuracy                         0.9938     13163
   macro avg     0.9580    0.9096    0.9324     13163
weighted avg     0.9936    0.9938    0.9937     13163

PR-AUC: 0.9995831433246334
ROC-AUC: 0.9857571308099646
```

seed 50
```
precision    recall  f1-score   support

           0     0.9328    0.6852    0.7900       324
           1     0.9921    0.9988    0.9954     12839

    accuracy                         0.9910     13163
   macro avg     0.9624    0.8420    0.8927     13163
weighted avg     0.9906    0.9910    0.9904     13163

PR-AUC: 0.9992194394255228
ROC-AUC: 0.974854657731699
```